In [9]:
from typing import Optional
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.pydantic_v1 import BaseModel, Field
from langchain_groq import ChatGroq

model = ChatGroq(temperature=2)


# Define your desired data structure.
class Joke(BaseModel):
    setup: str = Field(description="question to set up a joke")
    punchline: str = Field(description="answer to resolve the joke")
    rating: Optional[int] = Field(
        description="How funny the joke is, from 1 to 10"
    )


# And a query intented to prompt a language model to populate the data structure.
joke_query = "Tell me a joke."

# Set up a parser + inject instructions into the prompt template.
parser = JsonOutputParser(pydantic_object=Joke)

prompt = PromptTemplate(
    template="Answer the user query.\n{format_instructions}\n{query}\n",
    input_variables=["query"],
    partial_variables={
        "format_instructions": parser.get_format_instructions(),
    },
)

chain = prompt | model | parser

chain.invoke(
    {
        "query": joke_query,
    }
)

{'setup': "Why don't scientists trust atoms?",
 'punchline': "'Cause they make up everything!'",
 'rating': 9}

In [10]:
parser.get_format_instructions()

'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"setup": {"title": "Setup", "description": "question to set up a joke", "type": "string"}, "punchline": {"title": "Punchline", "description": "answer to resolve the joke", "type": "string"}, "rating": {"title": "Rating", "description": "How funny the joke is, from 1 to 10", "type": "integer"}}, "required": ["setup", "punchline"]}\n```'

In [11]:
for s in chain.stream(
    {
        "query": joke_query,
    }
):
    print(s)

{}
{'setup': ''}
{'setup': 'Why'}
{'setup': 'Why don'}
{'setup': "Why don'"}
{'setup': "Why don't"}
{'setup': "Why don't scientists"}
{'setup': "Why don't scientists trust"}
{'setup': "Why don't scientists trust atoms"}
{'setup': "Why don't scientists trust atoms?"}
{'setup': "Why don't scientists trust atoms?", 'punchline': "'"}
{'setup': "Why don't scientists trust atoms?", 'punchline': "'C"}
{'setup': "Why don't scientists trust atoms?", 'punchline': "'Cause"}
{'setup': "Why don't scientists trust atoms?", 'punchline': "'Cause they"}
{'setup': "Why don't scientists trust atoms?", 'punchline': "'Cause they make"}
{'setup': "Why don't scientists trust atoms?", 'punchline': "'Cause they make up"}
{'setup': "Why don't scientists trust atoms?", 'punchline': "'Cause they make up everything"}
{'setup': "Why don't scientists trust atoms?", 'punchline': "'Cause they make up everything!"}
{'setup': "Why don't scientists trust atoms?", 'punchline': "'Cause they make up everything!", 'rating': 